### Step 1 - Read csv file

Take all environment varibles from below commnd

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config


In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
sources_path = f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType
schema=StructType([
    StructField('circuitId',StringType()),
    StructField('url',StringType()),
    StructField('circuitName',StringType()),
    StructField('lat',DoubleType()),
    StructField('long',DoubleType()),
    StructField('locality',StringType()),
    StructField('country',StringType())
    ]
)


In [0]:
circuit_df=(
    spark.read.format('csv')
    .option('header',True)
    #.option('inferSchema',True)
    .option('mode','FAILFAST')
    .schema(schema)
    .load(sources_path))

In [0]:
#display(circuit_df)

### Step 2 - Ingest time stamp and file path

In [0]:
from pyspark.sql.functions import *


In [0]:
circuit_final_df = add_ingestion_metadata(circuit_df)


### Step3- Write data to Bronze delta table

In [0]:
#circuit_final_df = circuit_final_df.withColumn('batch_id',lit(v_batch_id))


In [0]:
'''(circuit_final_df
 .write
 .mode('overwrite')
 .format('delta')
 .partitionBy('batch_id')
 .option('replaceWhere',f"batch_id = '{v_batch_id}'")
 .saveAsTable(table_name)
)'''

In [0]:
write_data_to_bronze(circuit_final_df,table_name, v_batch_id)

In [0]:
%sql
--drop table if exists formula1_incr.bronze.circuits;

### Step4 - Read delta table using sql

In [0]:
%sql
select * from formula1_incr.bronze.circuits;